# core

> An async Jupyter kernel client over HTTP and websockets

In [ ]:
#| default_exp core

Jupyasyncclient controls kernels through the Jupyter kernels HTTP API and exchanges their messages over a websocket. It works with [rustygate](https://github.com/AnswerDotAI/rustygate), jupygate, and jupyter_server. HTTP handles kernel creation, interrupt, restart, and deletion. One websocket carries the legacy Jupyter message protocol, with a `channel` field identifying each message's channel.

There is no zmq client in this package. The gateway handles zmq subscriptions, socket identities, and kernel communication. The client's send loop awaits websocket sends.

If you've used jupyter_client's `AsyncKernelClient`, names such as `execute`, `kernel_info`, and `wait_for_ready` will look familiar. Choose how to receive an execution's result:

- `execute` queues the request and returns its `msg_id` without waiting for a reply.
- `reply` queues the request and returns an awaitable for its `execute_reply`.
- `run` returns an async generator of that execution's messages. It also accepts a handler for that execution's stdin prompts.

Generated `*_request` methods provide access to protocol requests without a dedicated wrapper. Every inbound message also reaches `on_jmsg` once, in websocket arrival order, after request routing. Each includes its `channel`. The callback can be synchronous or asynchronous. An awaitable return completes before the reader takes the next message. Do not await a reply on the same websocket from the callback. An exception from the callback is logged with its traceback and the receive loop continues. Consumers that prefer queues can attach `JmsgQueues`. We'll use its merged `jmsg` queue below.

The examples start a disposable rustygate server and an [ipymini](https://github.com/AnswerDotAI/ipymini) kernel. The standard protocol operations also work with jupyter_server. Features such as notebook bindings and kernmini's held executions require the corresponding gateway or kernel extensions.

In [ ]:
#| export
import asyncio, functools, inspect, json, logging, os, ssl, time, uuid, websockets
from fasttransport.core import AsyncTransport
from fastspec.oapi import OpenAPIClient, SpecParser
from fasttransport.errors import APIError
from contextlib import suppress
from urllib.parse import urlencode, urlsplit, urlunsplit
from jupywire.session import Session, validate_string_dict, dumps, loads, serialize_binary_message, deserialize_binary_message
from jupywire.ops import EvalOps
from jupywire.route import RouterOps, DeadKernelError
from fastcore.basics import patch
from fastcore.xtras import dict2obj
from fastcore.net import urlread


In [ ]:
from fastcore.test import test_eq, test_fail, ExceptionExpected
from fastcore.nbio import msg2out
from jupywire.route import JmsgQueues, OUTPUT_MSGS, COMM_MSGS
from jupywire.ops import EvalError
from rustygate.tools import start_gateway
from pathlib import Path


In [ ]:
#| export
log = logging.getLogger('jupyasyncclient')

## Wire codec

Legacy Jupyter websockets use JSON text frames for messages without buffers. Binary frames contain a count, an offset table, the JSON message, then its raw buffers. The [jupygate docs](https://AnswerDotAI.github.io/jupygate/core.html) describe the byte layout.

We use `jupywire.session`'s `dumps`, `loads`, `serialize_binary_message`, and `deserialize_binary_message`. Binary decoding returns `memoryview` buffers without copying their bytes. This example checks both encodings:

In [ ]:
ses = Session(key=b'demo')
msg = ses.msg('comm_msg', dict(comm_id='c', data={}))
msg['channel'] = 'iopub'
msg['buffers'] = [b'raw']
back = deserialize_binary_message(serialize_binary_message(msg))
test_eq(bytes(back['buffers'][0]), b'raw')
test_eq(loads(dumps(dict(msg, buffers=[])))['content'], msg['content'])
back['channel']

'iopub'

## Shared HTTP plumbing

`KernelApi` provides the HTTP operations shared by this package's three public client classes. It joins URL paths and converts `http` to `ws`, or `https` to `wss`, for websocket endpoints. It also prepares token headers and an [AsyncTransport](https://github.com/AnswerDotAI/fasttransport). The transport can use an HTTP client you supply. Otherwise it creates a client for each request.

The `.api` methods come from [fastspec](https://github.com/AnswerDotAI/fastspec). They use a compact, pre-parsed copy of rustygate's OpenAPI specification in `rg_spec.py`. All operations share the client's transport. Their signatures and documentation come from the specification.

The standard kernel operations work with jupyter_server too. Rustygate-specific endpoints require rustygate.

In [ ]:
#| export
def _join_url(base, path, ws=False, params=None):
    u = urlsplit(base)
    scheme = {"http": "ws", "https": "wss"}.get(u.scheme, u.scheme) if ws else u.scheme
    base_path = u.path.rstrip("/")
    full_path = f"{base_path}/{path.lstrip('/')}" if path else base_path
    query = urlencode({k: v for k, v in (params or {}).items() if v is not None})
    return urlunsplit((scheme, u.netloc, full_path, query, ""))

In [ ]:
#| export
@functools.cache
def rg_spec():
    "The bundled rustygate spec, loaded once per process from its compact `rg_spec` module."
    from jupyasyncclient.rg_spec import spec
    return SpecParser.from_dict(spec)

def build_spec(nm='jupyasyncclient/rg_spec.py', url='http://localhost:8787'):
    "Regenerate the compact spec module `rg_spec.py` from a running rustygate's `/openapi.json`."
    SpecParser.from_openapi(dict2obj(json.loads(urlread(f'{url}/openapi.json')))).save(nm)

class KernelApi:
    "Shared HTTP plumbing for the Jupyter kernels API: one transport, and the spec-generated ops on `api`."
    def __init__(self, base_url, token=None, headers=None, timeout=30, http_client=None, verify=True):
        self.base_url,self.token,self._timeout,self.verify = base_url.rstrip('/'),token or '',timeout,verify
        self._headers = {**(headers or {})}
        if self.token and 'Authorization' not in self._headers: self._headers['Authorization'] = f'token {self.token}'
        self.transport = AsyncTransport(timeout=timeout, client=http_client, base_headers=self._headers, verify=verify)
        self.api = OpenAPIClient(rg_spec(), transport=self.transport, base_url=self.base_url)

    def _ws_ssl(self, url):
        "An unverified ssl context when `verify=False` and `url` is wss, else None for the library default."
        if self.verify or not url.startswith('wss'): return None
        ctx = ssl.create_default_context()
        ctx.check_hostname,ctx.verify_mode = False,ssl.CERT_NONE
        return ctx

    def _kpath(self, kernel_id='', suffix=''): return f"/api/kernels/{kernel_id}{suffix}" if kernel_id else '/api/kernels'

## The client

`JupyAsyncKernelClient` manages one kernel's HTTP operations, websocket, and protocol messages. You can make HTTP calls before opening its websocket. Constructing the client does not create a kernel or open channels.


In [ ]:
#| export
class JupyAsyncKernelClient(RouterOps, EvalOps, KernelApi):
    "AsyncKernelClient-ish API over the kernels HTTP API plus its websocket channels."
    allow_stdin = True
    def __init__(self, base_url, kernel_id=None, token=None, session_id=None, username=None, headers=None, timeout=30, http_client=None,
        reconnect=True, reconnect_ceiling=300.0, verify=True, max_size=256*2**20):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify)
        self.kernel_id = kernel_id
        self.owned = False    # True only when `start_kernel` created the kernel; honored by `__aexit__`
        self.session_id = session_id or uuid.uuid4().hex
        self.session = Session(session=self.session_id, username=username or os.environ.get("USER") or "")
        self._ws,self._start_task,self._send_task,self._recv_task,self._close_task = [None]*5
        self.reconnect,self.reconnect_ceiling,self._unsent,self._closing = reconnect,reconnect_ceiling,None,False
        self.max_size = max_size
        self._send_q = asyncio.Queue()
        self._init_router()

    def _kpath(self, kernel_id='', suffix=''): return super()._kpath(kernel_id or self.kernel_id, suffix)

`aclose` stops the client's background tasks, closes its websocket, and fails pending replies and runs. It leaves the kernel running. `stop_channels` schedules this cleanup without awaiting it. That method still requires a running event loop.

In [ ]:
#| export
@patch
async def aclose(self: JupyAsyncKernelClient):
    self._closing = True
    if self._send_task and not self._send_task.done():   # the None sentinel ends `_send_loop` once the queue has drained
        self._send_q.put_nowait(None)
        with suppress(asyncio.CancelledError, Exception):
            async with asyncio.timeout(2): await self._send_task
    for t in (self._start_task, self._send_task, self._recv_task):
        if t and not t.done(): t.cancel()
    if self._ws and self._ws.close_code is None: await self._ws.close()
    for t in (self._start_task, self._send_task, self._recv_task):
        if t:
            with suppress(asyncio.CancelledError, Exception): await t
    self.fail_waiters(RuntimeError('client closed'))
    self._ws = None

@patch
def stop_channels(self: JupyAsyncKernelClient):
    if self._close_task and not self._close_task.done(): return
    self._close_task = asyncio.create_task(self.aclose())

In [ ]:
#| export
@patch
async def start_kernel(self:JupyAsyncKernelClient, kernel_name="py", **kwargs):
    for k in ('path', 'cwd'):
        if kwargs.get(k) is not None: kwargs[k] = str(kwargs[k])
    response = await self.api.kernels.create_kernel(name=kernel_name, raw_=True, **kwargs)
    model = dict2obj(response.json())
    self.kernel_id,self.owned = model['id'],response.status_code == 201
    return model

`start_kernel` posts to `/api/kernels`, stores the returned kernel id, and returns the model. It converts non-`None` `path` and `cwd` arguments to strings before sending them. `None` remains unchanged. A 201 response sets `owned=True`; a 200 response reuses an existing kernel and sets `owned=False`. With rustygate, `path` selects the shared kernel for that notebook. Let's start a local gateway and create a kernel:


In [ ]:
g = start_gateway()
base_url = g.url
g


<Gateway http://127.0.0.1:60485 pid=47780 up>

In [ ]:
kc = JupyAsyncKernelClient(base_url)
model = await kc.start_kernel()
model

```python
{ 'connections': 0,
  'execution_state': 'alive',
  'id': '835147e537bb42e990b74a75a4bf9d1e',
  'language': 'python',
  'last_heartbeat': None,
  'name': '',
  'path': None,
  'pid': 47784}
```

`shutdown_kernel`, `interrupt_kernel`, and `restart_kernel` call the corresponding HTTP endpoints. They do not require open websocket channels. `shutdown_kernel` also closes this client, even if the HTTP deletion fails.

In [ ]:
#| export
@patch
async def shutdown_kernel(self: JupyAsyncKernelClient):
    "Request kernel deletion, then close this client even if deletion fails."
    try:
        if self.kernel_id: return await self.api.kernels.delete_kernel(kid=self.kernel_id)
    finally: await self.aclose()
@patch
async def interrupt_kernel(self: JupyAsyncKernelClient):
    if self.kernel_id: return await self.api.kernels.interrupt(kid=self.kernel_id)
@patch
async def restart_kernel(self: JupyAsyncKernelClient):
    if self.kernel_id: return await self.api.kernels.restart(kid=self.kernel_id)


`model` returns the gateway's kernel model. Rustygate includes `execution_state`, `pid`, and the bound notebook `path`; other servers can return different fields.

`is_alive` reports whether the kernel is available. It returns `False` without a kernel id, for a dead kernel, or when it cannot fetch the model. Unknown ids produce HTTP 404.
Busy, idle, and unresponsive kernels count as alive. A nonempty model without `execution_state` also counts as alive, for compatibility with other servers.

In [ ]:
#| export
@patch
async def model(self: JupyAsyncKernelClient):
    "What the gateway reports about this kernel (jupyter's `kernel model`): `execution_state`, `pid`, `path`, ..."
    return await self.api.kernels.get_kernel(kid=self.kernel_id)

@patch
async def is_alive(self: JupyAsyncKernelClient):
    if not self.kernel_id: return False
    try: return bool(m := await self.model()) and m.get('execution_state') != 'dead'
    except Exception: return False

In [ ]:
m = await kc.model()
test_eq((m['id'], await kc.is_alive()), (kc.kernel_id, True))
none_kc = JupyAsyncKernelClient(base_url)
test_eq(await none_kc.is_alive(), False)
m

```python
{ 'connections': 0,
  'execution_state': 'alive',
  'id': '835147e537bb42e990b74a75a4bf9d1e',
  'language': 'python',
  'last_heartbeat': 1789266815.870322,
  'name': '',
  'path': None,
  'pid': 47784}
```

Use `.api` for operations without a convenience method, including rustygate's search and exec endpoints. Displaying an operation shows the signature and parameter documentation from the bundled specification:

In [ ]:
test_eq((await kc.api.kernels.get_kernel(kid=kc.kernel_id))['id'], kc.kernel_id)
kc.api.kernels.restart

`await kernels.restart(kid)`

Restart the kernel in place, keeping its id and its clients' websockets; returns the new model.

Parameters:
- kid (str, required): Kernel id

Call options (in addition to schema parameters): `headers_` adds HTTP headers; `query_` merges extra query entries; `body_` merges extra body fields; `raw_=True` returns the transport response without decoding it.

`stream=True` returns an async iterator after awaiting the call. It cannot be combined with `raw_=True`.

HTTP errors raise `fasttransport.errors.APIError`; inspect its `status_code` and message.

In [ ]:
#| export
@patch
async def kernel_for(self: KernelApi, path):
    "The non-dead kernel model bound to notebook `path`, or None"
    ms = await self.api.kernels.list_kernels(path=str(path))
    return next((m for m in ms if m['execution_state'] != 'dead'), None)

`kernel_for(path)` asks rustygate for kernels bound to that notebook path and returns the first model whose `execution_state` is not `dead`. It returns `None` when none match. A dead kernel can retain its binding in the registry without counting as a live match.

This method belongs to `KernelApi` and calls the list endpoint. It does not depend on the calling client's `kernel_id`.

In [ ]:
b = JupyAsyncKernelClient(base_url)
await b.start_kernel(path='bound.ipynb')
found = await kc.kernel_for('bound.ipynb')
test_eq(found['id'], b.kernel_id)
test_eq(await kc.kernel_for('unbound.ipynb'), None)
await b.shutdown_kernel()

### How it works

`jupywire.route.RouterOps` handles messages for both this client and `conkernelclient`. Jupywire's `DESIGN.md` explains the shared routing contract.

The receive loop passes each decoded message to `route`. A run collects messages with its request id in `parent_header.msg_id`. Its stdin prompts are answered by its `on_stdin` handler. A shell or control message can resolve a pending reply with the same parent id. Every message also reaches `on_jmsg` once after this routing. Notification does not transfer stdin ownership.

A `status` with `execution_state='dead'` takes precedence over those routes. It fails pending replies and runs, then reaches `on_jmsg`. The router also remembers stdin request headers for `input` replies.


In [ ]:
#| export
@patch
async def _ensure_started(self: JupyAsyncKernelClient):
    if not self._start_task: self.start_channels()
    if self._start_task: await self._start_task

@patch
def send(self: JupyAsyncKernelClient, msg, channel: str):
    "Serialize `msg` (binary framing when it carries buffers) and queue its frame for `_send_loop`; returns the msg_id."
    msg = dict(msg)
    msg["channel"] = channel
    bufs = msg.get("buffers")
    if bufs:
        msg["buffers"] = [bytes(b) for b in bufs]
        payload = serialize_binary_message(msg)
    else:
        msg.pop("buffers", None)
        payload = dumps(msg)
    self._send_q.put_nowait(payload)
    return msg["header"]["msg_id"]

In [ ]:
#| export
@patch
async def _send_loop(self: JupyAsyncKernelClient):
    assert self._ws is not None
    while True:
        payload = self._unsent if self._unsent is not None else await self._send_q.get()
        if payload is None: return
        self._unsent = payload
        try: await self._ws.send(payload)
        except Exception as e:
            log.warning("websocket send failed: %s", e)
            return  # the payload stays in `_unsent`: a reconnected send loop retries it first
        self._unsent = None

@patch
async def _recv_loop(self: JupyAsyncKernelClient):
    assert self._ws is not None
    with suppress(websockets.ConnectionClosed):
        async for data in self._ws:
            if isinstance(data, str): msg = loads(data)
            elif isinstance(data, bytes): msg = deserialize_binary_message(data)
            else: continue
            try:
                r = self.route(msg)
                if inspect.isawaitable(r): await r
            except Exception: log.exception('inbound message handler failed')
    if not self._closing:
        if self.reconnect: self._start_task = asyncio.create_task(self._reconnect())
        else: self.fail_waiters(ConnectionError('websocket closed (reconnect disabled)'))

`_start_ws` connects to the kernel's channels endpoint and starts the send and receive tasks. It cancels old tasks before replacing the connection:

In [ ]:
#| export
@patch
async def _start_ws(self: JupyAsyncKernelClient):
    if self._ws and self._ws.close_code is None: return
    for t in (self._send_task, self._recv_task):
        if t and not t.done(): t.cancel()  # a stale send loop must not steal payloads from the new one
    params = {"session_id": self.session_id}
    if self.token: params["token"] = self.token
    ws_url = _join_url(self.base_url, self._kpath(suffix="/channels"), ws=True, params=params)
    self._ws = await websockets.connect(ws_url, ssl=self._ws_ssl(ws_url), additional_headers=self._headers, ping_interval=30, max_size=self.max_size)
    self._send_task = asyncio.create_task(self._send_loop())
    self._recv_task = asyncio.create_task(self._recv_loop())

`_exec_req` builds the message header and content, then calls `send` to serialize and queue it. It returns the message id. Websocket messages have no HMAC signature; the gateway handles signing when it forwards them over zmq.

Queue order follows call order. The send loop transmits frames later, independently of when callers await their replies.

`__getattr__` creates senders for names ending in `*_request`. Each returns an awaitable for a reply. This makes new request types accessible without adding wrappers here, but it does not check whether the kernel supports a name. Other missing attributes raise `AttributeError`. `RouterOps` implements `reply` and `run` using `execute` and `send`.


In [ ]:
#| export
@patch
def _exec_req(self: JupyAsyncKernelClient, name, content=None, channel="shell", metadata=None, subshell_id=None, parent=None, msg_id=None, buffers=None):
    "Build and queue a protocol message without waiting for its reply; return its msg_id."
    msg = self.session.msg(name, content, metadata=metadata, parent=parent)
    if buffers: msg["buffers"] = buffers
    if subshell_id: msg["header"]["subshell_id"] = subshell_id
    if msg_id: msg["header"]["msg_id"] = msg_id
    return self.send(msg, channel)

def _gen_request(self, name):
    "Create an awaitable reply sender for a public `*_request` name. Other names raise AttributeError. Assign this to the class: module-level __getattr__ would invoke PEP 562 instead."
    if name.startswith("_") or not name.endswith("_request"): raise AttributeError(name)
    def _f(channel="shell", timeout=None, **kwargs): return self.request(name, kwargs or None, channel, timeout=timeout)
    return _f

JupyAsyncKernelClient.__getattr__ = _gen_request

### Connecting

In [ ]:
#| export
@patch
def start_channels(self: JupyAsyncKernelClient, shell=True, iopub=True, stdin=True, control=True):
    if not (shell or iopub or stdin or control): return self
    if self._start_task and not self._start_task.done(): return self
    self._start_task = asyncio.create_task(self._start_ws())
    return self

In [ ]:
#| export
@patch(as_prop=True)
def channels_running(self: JupyAsyncKernelClient): return bool(self._ws and self._ws.close_code is None)

`start_channels` schedules the websocket connection and returns the client immediately, including when no connection needs to be started. It supplies `session_id` and, when configured, `token` as query parameters. `channels_running` reports whether the websocket is open. Await `wait_for_ready` before using a newly connected client.


Without an `on_jmsg` callback, the client retains no traffic beyond active request routing. Attach `JmsgQueues` to observe all inbound traffic through queues. It installs itself as the callback and routes messages according to their channel. A timed-out read raises `queue.Empty`.

The examples merge iopub, stdin, and `cells` into one `jmsg` queue. Rustygate uses the nonstandard `cells` channel for notebook change broadcasts named `cell_ops`. The files notebook demonstrates those messages.


In [ ]:
#| export
@patch
async def wait_for_ready(self: JupyAsyncKernelClient, timeout=None):
    await self._ensure_started()
    await self.kernel_info_request(timeout=timeout)
    return self

`wait_for_ready` opens the channels if needed and awaits one `kernel_info_request` reply. It returns the client when ready. The gateway handles its own zmq subscription readiness. The client does not drain iopub while waiting.

That matters on reconnection: draining could throw away output the gateway replays for this `session_id`. Replayed output and the readiness request's status messages follow the normal routes. An `on_jmsg` handler must tolerate message types it doesn't use.

This method calls the generated request sender, without depending on the `kernel_info` wrapper defined below.


In [ ]:
assert kc.start_channels(shell=False, iopub=False, stdin=False, control=False) is kc
assert kc.start_channels().start_channels() is kc
assert await kc.wait_for_ready(timeout=60) is kc
kc.channels_running


True

You can await a generated sender directly. Here is `comm_info_request`; the `comm_info` convenience method appears later:


In [ ]:
m = await kc.comm_info_request(timeout=15)
m['msg_type']


'comm_info_reply'

### The request API

In [ ]:
#| export
@patch
def execute(self: JupyAsyncKernelClient, code, silent=False, store_history=True, user_expressions=None, allow_stdin=None, stop_on_error=True,
    msg_id=None, metadata=None, subshell_id=None, buffers=None):
    "Queue an `execute_request` without waiting for a reply; return its msg_id."
    user_expressions = {} if user_expressions is None else user_expressions
    allow_stdin = self.allow_stdin if allow_stdin is None else allow_stdin
    if not isinstance(code, str): raise ValueError(f"code {code!r} must be a string")
    validate_string_dict(user_expressions)
    return self._exec_req("execute_request", dict(code=code, silent=silent, store_history=store_history, user_expressions=user_expressions,
        allow_stdin=allow_stdin, stop_on_error=stop_on_error), metadata=metadata, subshell_id=subshell_id, msg_id=msg_id, buffers=buffers)

Let's attach `JmsgQueues` for the remaining examples. A fire-and-forget `execute` has no reply waiter or run to collect its messages. Its printed output reaches the merged queue:


In [ ]:
qs = JmsgQueues(kc, queues=('shell', 'control', 'jmsg'), merge=dict(iopub='jmsg', stdin='jmsg', cells='jmsg'))
kc.execute("print('unread')")
s = await qs.jmsg_for('stream', timeout=15)
s['content']['text']


'unread\n'

Awaiting `reply` gives you the `execute_reply`. Its iopub output still arrives separately through `on_jmsg`, which our queue adapter collects:


In [ ]:
rep = await kc.reply("print('hello'); 6*7", timeout=30)
test_eq(rep['content']['status'], 'ok')
s = await qs.jmsg_for('stream', timeout=15)
r = await qs.jmsg_for('execute_result', timeout=15)
dict(stream=s['content']['text'].strip(), result=r['content']['data']['text/plain'])


{'stream': 'hello', 'result': '42'}

Use `pred` to select more than a message type. This reads until it finds an `idle` status. `jmsg_for` discards messages that don't match:

In [ ]:
idle = await qs.jmsg_for('status', pred=lambda m: m['content']['execution_state']=='idle', timeout=15)
idle['content']['execution_state']


'idle'

Several requests can wait for replies at once. Each future matches its own request id. Calls queue requests immediately, before you await their results. Await order does not change submission order:

In [ ]:
reps = await asyncio.gather(*[kc.reply(f'{i}*{i}', timeout=30) for i in range(5)])
[r['content']['status'] for r in reps]


['ok', 'ok', 'ok', 'ok', 'ok']

Pass `msg_id` to `execute`, `reply`, or `run` to choose the request's header id. The kernel copies it into the parent header of messages caused by that request. Hosts use this to associate output with their own work, such as solveit's `{message_id}.{token}` ids. Use distinct ids for requests that are in flight together:

In [ ]:
rep = await kc.reply("21*2", timeout=30, msg_id='myid.abc123')
test_eq(rep['parent_header']['msg_id'], 'myid.abc123')


A timeout or cancellation removes a pending reply's entry. It does not stop the kernel execution. If the reply arrives later, no waiter claims it and the router passes it to `on_jmsg`:

In [ ]:
lid = f'late.{uuid.uuid4().hex[:6]}'
w = kc.reply('import time; time.sleep(1)', timeout=0.25, msg_id=lid)
with ExceptionExpected(TimeoutError): await w
assert lid not in kc.replies
late = await qs.jmsg_for('execute_reply', queue='shell', pred=lambda m: m['parent_header']['msg_id']==lid, timeout=15)
late['msg_type']

'execute_reply'

These methods send JEP 91 subshell requests on the control channel. Ipymini accepts a stable `subshell_id`: creating it again returns the same subshell. Omitting the id creates a new one. `list_subshell` returns the live ids.

The example starts work in a subshell and requests completion from the main shell while that work continues. These behaviors require kernel support.

In [ ]:
#| export
@patch
def create_subshell(self: JupyAsyncKernelClient, subshell_id=None, timeout=None):
    return self.create_subshell_request(subshell_id=subshell_id, channel="control", timeout=timeout)

@patch
def list_subshell(self: JupyAsyncKernelClient, timeout=None): return self.list_subshell_request(channel="control", timeout=timeout)

@patch
def delete_subshell(self: JupyAsyncKernelClient, subshell_id: str, timeout=None):
    return self.delete_subshell_request(subshell_id=subshell_id, channel="control", timeout=timeout)

In [ ]:
sub = (await kc.create_subshell('lesson-sidecar', timeout=15))['content']['subshell_id']
same = (await kc.create_subshell('lesson-sidecar', timeout=15))['content']['subshell_id']
test_eq((sub, same), ('lesson-sidecar', 'lesson-sidecar'))
assert sub in (await kc.list_subshell(timeout=15))['content']['subshell_id']
await kc.complete('imp', timeout=15)  # the first completion is slow, so it runs before the timed one
busy = kc.reply('import time; time.sleep(1.5)', timeout=30, subshell_id=sub)
matches,_ = await kc.complete('imp', timeout=1)
rep = await busy
await kc.delete_subshell(sub, timeout=15)
rep['content']['status'], rep['parent_header']['subshell_id']==sub, 'import' in matches


('ok', True, True)

In [ ]:
#| export
@patch
def release(self: JupyAsyncKernelClient, msg_id: str, status: str = "ok", timeout=None):
    "Complete a held execute (kernmini's `hold` metadata); `status='error'` makes the hold's reply an error, engaging the kernel's stop-on-error tail abort"
    return self.release_request(msg_id=msg_id, status=status, channel="control", timeout=timeout)

Kernmini's `hold` execute metadata pauses ordinary queued work while another application does work outside the kernel. Requests with numeric `priority` metadata can run ahead of that queue. Higher priorities run first, with FIFO order within one priority.

The gateway forwards this metadata unchanged. `release` sends a control request to finish the hold. Its reply's `found` field says whether that hold still existed. Releasing an expired hold has no effect:

In [ ]:
hid = f'hold1.{uuid.uuid4().hex[:6]}'
kc.execute('', metadata=dict(hold=True), msg_id=hid)
tail = kc.reply("'after the hold'", timeout=30)
jump = await kc.reply("'jumped'", timeout=30, metadata=dict(priority=1))
test_eq(jump['content']['status'], 'ok')
rel = await kc.release(hid, timeout=15)
test_eq(rel['content']['found'], True)
test_eq((await tail)['content']['status'], 'ok')


Releasing a hold with `status='error'` triggers the kernel's stop-on-error behavior. Queued requests finish with `status='aborted'`, including replies that callers are awaiting.

Shell and control messages use separate zmq channels at the gateway. The example pauses briefly to give the shell request time to queue before the control release arrives:

In [ ]:
hid = f'hold2.{uuid.uuid4().hex[:6]}'
kc.execute('', metadata=dict(hold=True), msg_id=hid)
t2 = kc.reply("'never runs'", timeout=30)
await asyncio.sleep(0.2)
await kc.release(hid, status='error', timeout=15)
test_eq((await t2)['content']['status'], 'aborted')


`input` answers the latest remembered stdin prompt and copies its header into the reply's `parent_header`:

In [ ]:
fut = asyncio.ensure_future(kc.reply("name = input('who? ')", allow_stdin=True, timeout=30))
prompt = await qs.jmsg_for('input_request', timeout=15)
kc.input('Jeremy')
await fut
rep = await kc.reply('name', timeout=30, user_expressions={'v':'name'})
rep['content']['user_expressions']['v']['data']['text/plain']


"'Jeremy'"

Generated `*_request` methods return full reply messages. For a name held in a variable, use `request(name, ...)`. `shell` and `control` select the channel explicitly.

`RouterOps` also provides conveniences for frontends. `complete` returns completion matches and the replacement start position. `inspect` returns text, or an empty string when nothing matches. Both default the cursor to the end of the code. `check` reports completeness and indentation; `history` returns a history reply. The same methods work in `conkernelclient`. Its comm senders also come from this mixin, but comm messages have no reply.


In [ ]:
matches, start = await kc.complete('imp')
assert 'import' in matches
matches[:3]


['import']

In [ ]:
txt = await kc.inspect('print')
assert 'print' in txt
test_eq(await kc.inspect('no_such_name_here'), '')


In [ ]:
h = await kc.history(timeout=15)
len(h['content']['history']) > 0

True

`comm_info` and `kernel_info` return the full protocol replies. `check` uses `is_complete_request` to help a frontend decide whether the user has finished typing a cell:


In [ ]:
#| export
@patch
def comm_info(self: JupyAsyncKernelClient, target_name=None, timeout=None):
    return self.comm_info_request(target_name=target_name, timeout=timeout)

@patch
def kernel_info(self: JupyAsyncKernelClient, timeout=None): return self.kernel_info_request(timeout=timeout)

In [ ]:
test_eq(await kc.check('for i in range(3):'), ('incomplete', '    '))
test_eq((await kc.check('1+1'))[0], 'complete')


### Calling kernel functions

`eval` calls `func(*args, **kw)` inside the kernel and waits if the result is a coroutine. It retrieves the result through `user_expressions` in an execution reply. The result travels as a repr, not as the original object.

By default, `try_eval` parses that repr when possible. For supported primitive results, it attempts to wrap the value in a class with the kernel-side type's name. A value it cannot reconstruct stays a string. `literal_=False` returns the representation without parsing. `call_=False` treats `func` as an expression instead of calling it.

`ipy` calls methods on `get_ipython()`. Service methods such as `sig_help`, `get_schemas`, and `ranked_complete` call the methods that `ipyfuncs` adds to the shell. This family comes from jupywire's `EvalOps` mixin, shared with `conkernelclient`.

Ordinary `eval` uses the main shell. With `sidecar_=True`, it uses kernmini's persistent subshell named `sidecar`, which the kernel creates on first use. Service methods and variable operations (`xpush`, `get_vars`, `eval_exprs`, and `retr`) default to that serial subshell. The numeric `priority` metadata shown above changes queue order; it does not select the sidecar.

In [ ]:
r = await kc.reply('def add(a, b): return a+b')
test_eq(r['content']['status'], 'ok')
test_eq(await kc.eval('add', a=10, b=20), 30)
await kc.reply('a = [1,2,3]')
test_eq(await kc.eval('a', call_=False), [1,2,3])
test_eq(await kc.eval('add', a=30, b=40, literal_=False), '70')
_r = await kc.eval('missing_fn')
assert 'NameError' in _r

`eval_expr` evaluates an expression through `user_expressions`. It parses the result with `literal_eval` when possible and otherwise returns the repr string. A kernel error raises `EvalError`. In contrast, `eval` returns error details, usually a traceback, as text:

In [ ]:
test_eq(await kc.eval_expr('a[-1] * 2'), 6)
with ExceptionExpected(EvalError, 'NameError'): await kc.eval_expr('no_such_name')


Import [ipyfuncs](https://github.com/AnswerDotAI/ipyfuncs) inside the kernel before calling its `_ipy_funcs` services. These examples include that setup:

In [ ]:
await kc.reply('import ipyfuncs')
await kc.reply('''def range_ex(
    a:str  # some param
):
    "some func docstring"
    ...''')
sigs = await kc.sig_help(code='range_ex(', line_no=1, col_no=9)
test_eq(sigs[0]['label'], 'range_ex')
schemas = await kc.get_schemas(fs=['range_ex'])
test_eq(schemas['range_ex']['function']['name'], 'range_ex')


`xpush`, `retr`, and `eval_exprs` use the serial sidecar by default. `xpush` queues assignments without waiting for a reply. `xenv` also returns without waiting, but sets environment variables through the main shell:

In [ ]:
kc.xpush(asdf=4)
test_eq(await kc.retr('asdf'), 4)
test_eq(await kc.eval_exprs(vs=['list(range(5))']), {'list(range(5))': [0,1,2,3,4]})
kc.xenv(hi='jupy')
test_eq(await kc.eval('__os.environ["hi"]', call_=False), 'jupy')

### Streaming an execution: `run`

Use `run` when you want one execution's messages as they arrive. A websocket also carries output from other executions, and those messages can arrive between yours. The run collects messages whose parent id matches its request id.

Like `reply`, `run` registers its request and queues it at call time. It returns an async generator of that execution's outputs, statuses, and `execute_reply`. The generator finishes after both the reply and the matching `idle` status arrive. It does not yield stdin requests: those go to the run's `on_stdin` handler. Without that handler, the execution disables stdin.

`exec_outs` collects a run's output messages into a list of nbformat outputs using `msg2out`. `RouterOps` provides both methods. Concurrent runs collect their own traffic. A dead-kernel status makes all waiting runs raise `DeadKernelError`.


Here is the same execution as a message stream and as a list of notebook outputs:


In [ ]:
types = [m['msg_type'] async for m in kc.run("print('hi'); 6*7")]
test_eq([t for t in types if t in OUTPUT_MSGS], ['stream', 'execute_result'])
outs = await kc.exec_outs("print('hi'); 6*7")
test_eq([o['output_type'] for o in outs], ['stream', 'execute_result'])
outs


[{'output_type': 'stream', 'name': 'stdout', 'text': 'hi\n'},
 {'output_type': 'execute_result',
  'metadata': {'__type': 'int'},
  'data': {'text/plain': '42'},
  'execution_count': 20}]

Render messages inside the `async for` loop as they arrive. The loop body can await other work, as clikernel's stream worker does.

Close the generator if you stop reading early. Closing removes the run's routing entry; it does not interrupt the cell. The remaining messages then reach `on_jmsg`:

In [ ]:
gen = kc.run("print('early'); 6*7")
test_eq((await anext(gen))['msg_type'], 'execute_queued')
await gen.aclose()
test_eq(kc.runs, {})
s = await qs.jmsg_for('stream', timeout=15)
s['content']['text']

'early\n'

A cell error produces an `error` output and an `execute_reply` with error status. The run yields both, rather than raising the cell's exception in the client:


In [ ]:
msgs = [m async for m in kc.run('1/0')]
rep = next(m for m in msgs if m['msg_type'] == 'execute_reply')
test_eq(rep['content']['status'], 'error')
outs = [msg2out(m) for m in msgs if m['msg_type'] in OUTPUT_MSGS]
test_eq(outs[0]['ename'], 'ZeroDivisionError')


Pass `on_stdin` to handle prompts for one run. The callback receives the complete `input_request` and returns an answer. It can be synchronous or asynchronous. Jupywire sends an `input_reply` with the correct parent header:


In [ ]:
run_prompts = []
async def answer_stdin(msg):
    run_prompts.append(msg)
    return 'jupy'

async for _ in kc.run("nm = input('who? ')", on_stdin=answer_stdin): pass
name = await kc.retr('nm')
test_eq(name, 'jupy')
dict(prompt=run_prompts[0]['content']['prompt'], name=name)


{'prompt': 'who? ', 'name': 'jupy'}

`run` also collects comm messages parented to its execution. `COMM_MSGS` contains `comm_open`, `comm_msg`, and `comm_close`. These are protocol messages, not notebook outputs.

Comms let kernel code talk to its host application, as ipyai's `%ipyai` magic does. Comm messages also reach `on_jmsg`:


In [ ]:
msgs = [m async for m in kc.run("from comm import create_comm\ncm = create_comm('demo', data=dict(x=1))\ncm.send(dict(y=2))")]
comms = [(m['msg_type'], m['content'].get('data')) for m in msgs if m['msg_type'] in COMM_MSGS]
test_eq(comms[0], ('comm_open', dict(x=1)))
test_eq(comms[1], ('comm_msg', dict(y=2)))
test_eq([m for m in msgs if m['msg_type'] in OUTPUT_MSGS], [])


This example submits a separate execution before starting a run. Its output must not appear in the run, and its `idle` must not finish the run. The router sends both executions to `on_jmsg`, but each run collects only its own traffic. The same rule applies to another client's output or a host's silent bridge execution:


In [ ]:
foreign_id = kc.execute("'FOREIGN'")
outs = await kc.exec_outs("'MINE'")
test_eq(len(outs), 1)
assert 'MINE' in outs[0]['data']['text/plain'] and 'FOREIGN' not in str(outs)
foreign = await qs.jmsg_for('execute_result', pred=lambda m: m['parent_header']['msg_id']==foreign_id, timeout=15)
assert 'FOREIGN' in str(foreign['content'])


Each run registers its routing entry before queueing the request. Several runs on one client therefore collect independently. The test limits the combined wait to 30 seconds to catch a routing failure:


In [ ]:
outs = await asyncio.wait_for(asyncio.gather(kc.exec_outs('1+1'), kc.exec_outs("print('two')"), kc.exec_outs('3+3')), 30)
test_eq([o[0]['data']['text/plain'] for o in (outs[0], outs[2])], ['2', '6'])
test_eq(outs[1][0]['text'], 'two\n')


Submitting runs concurrently does not make the kernel execute their cells concurrently. They enter its queue. With the default `stop_on_error=True`, an error can abort queued executions with status `aborted`. Pass `stop_on_error=False` when you want later cells to run despite an earlier error:

In [ ]:
o1, o2, o3 = await asyncio.wait_for(asyncio.gather(*[kc.exec_outs(c, stop_on_error=False) for c in ('y = 6*7', '1/0', 'y')]), 30)
test_eq((o1, o2[0]['ename'], o3[0]['data']['text/plain']), ([], 'ZeroDivisionError', '42'))


## Protocol details

An execution reply can include payload dictionaries. Their `source` identifies what the frontend should do. Standard sources include `page` for pager content and `set_next_input` for preparing another input cell.

The aimagic nbextension extends `set_next_input` with `ctype` and `offset`. For example:

```py
pm = get_ipython().payload_manager
pm.write_payload(dict(
    source='set_next_input', text='bar',
    ctype='markdown', replace=False, offset=1))
```

You can use a custom source too. The kernel includes it in the reply; the frontend decides how to handle it. Pass `single=False` to keep multiple payloads with the same source:

In [ ]:
code = '''payl = dict(source='testing', foo='bar')
pm = get_ipython().payload_manager
pm.write_payload(payl, single=False)
pm.write_payload(dict(source='testing', foo='baz'), single=False)'''
r = await kc.reply(code, timeout=30)
test_eq([p['foo'] for p in r['content']['payload']], ['bar','baz'])

Pass `metadata` and `transient` to `display` to include them in a display message. Transient data describes the live display and should not become persistent notebook content:

In [ ]:
kc.execute("from IPython.display import display\ndisplay('hi', metadata={'key':'value'}, transient={'foo':'bar'})")
dd = await qs.jmsg_for('display_data', timeout=15)
test_eq(dd['content']['metadata'], {'key':'value'})
test_eq(dd['content']['transient'], {'foo':'bar'})

A `display_id` lets the frontend update an earlier display. The kernel sends `update_display_data` with the same id in `transient`. A frontend that tracks display ids replaces the corresponding content instead of appending a new output:

In [ ]:
code = '''from IPython.display import display
display('first', display_id='qqww')
display('second', update=True, display_id='qqww')'''
kc.execute(code)
upd = await qs.jmsg_for('update_display_data', timeout=15)
test_eq(upd['content']['transient'], {'display_id':'qqww'})
test_eq(upd['content']['data']['text/plain'], "'second'")

Interrupt through `interrupt_request` on control or through the HTTP `interrupt` endpoint. In this example, the running cell stops with a `KeyboardInterrupt` error output:

In [ ]:
task = asyncio.ensure_future(kc.exec_outs("import time; time.sleep(10); print('finished')"))
await asyncio.sleep(0.3)
await kc.interrupt_kernel()
outs = await task
test_eq(outs[0]['ename'], 'KeyboardInterrupt')


### Reconnecting

Zmq reconnects its sockets internally. This client must reconnect its websocket explicitly. When the receive loop ends after a closed connection, it starts `_reconnect` unless the client is closing or `reconnect=False`.

The new connection uses the same `session_id`. Pending replies and runs keep their routing entries, so messages arriving after reconnection still reach them. Rustygate does not replay messages missed during disconnection: a pending request can time out, and notebook clients must reload current state. Other gateways may buffer disconnected sessions.

The send loop retries `_unsent`, the frame whose send did not complete, before reading more queued frames. This is not an exactly-once execution guarantee.

After a failed connection attempt, `_reconnect` requests the kernel model over HTTP. HTTP `404` means the kernel no longer exists, and pending replies and runs fail with `DeadKernelError`. Other non-retryable HTTP errors propagate as `APIError`.

For retryable errors or an unreachable gateway, reconnection uses exponential backoff, starting at 0.1 seconds and capped at 5 seconds. Retries stop after `reconnect_ceiling`, which defaults to 300 seconds, and waiters fail with `ConnectionError`.

With `reconnect=False`, the client fails pending replies and runs when the connection closes instead of retrying.

In [ ]:
#| export
@patch
async def _reconnect(self: JupyAsyncKernelClient):
    "Redial with the same session id, with backoff; on a dead kernel or an expired ceiling, fail every pending reply and in-flight run."
    deadline, delay = time.monotonic() + self.reconnect_ceiling, 0.1
    while not self._closing:
        try:
            await self._start_ws()
            log.info("websocket reconnected")
            return
        except Exception as e:
            exc = None
            try: await self.model()
            except APIError as he:
                if he.status_code == 404: exc = DeadKernelError(f'kernel {self.kernel_id} is gone: {he.message}')
                elif he.status_code and not he.retryable: exc = he
            except Exception: pass  # the server is unreachable too: keep trying until the ceiling
            if exc is None and time.monotonic() > deadline: exc = ConnectionError(f'gave up reconnecting after {self.reconnect_ceiling}s: {e}')
            if exc:
                self.fail_waiters(exc)
                raise exc
            await asyncio.sleep(delay)
            delay = min(delay*2, 5.0)

In [ ]:
fut = asyncio.ensure_future(kc.reply("import time; time.sleep(1); 'survived'", timeout=20))
await asyncio.sleep(0.3)
kc._ws.transport.abort()
rep = await fut
test_eq(rep['content']['status'], 'ok')
test_eq((await kc.reply('1+1', timeout=15))['content']['status'], 'ok')
kc.channels_running


True

An active run keeps its routing entry across reconnection. Here the new connection opens before the sleeping execution finishes, so its later result still arrives:

In [ ]:
gen = kc.run("import time; time.sleep(1); 'collected'", timeout=30)
await anext(gen)
kc._ws.transport.abort()
msgs = [m async for m in gen]
test_eq(next(m['content']['status'] for m in msgs if m['msg_type'] == 'execute_reply'), 'ok')

Rustygate reports a kernel's process exit with a generated `status` whose `content.execution_state` is `dead`. The router fails pending replies and runs with `DeadKernelError`, calls the `on_dead(msg)` hook if one is set, then passes the status to `on_jmsg`. It needs that gateway message because the dead process cannot reply.

If the websocket closes first, a failed reconnect can detect an unavailable kernel through the HTTP model request described above. Here we deliberately kill a disposable test kernel during execution:

In [ ]:
d = JupyAsyncKernelClient(base_url)
await d.start_kernel()
d.start_channels()
await d.wait_for_ready(timeout=60)
deaths = []
d.on_dead = deaths.append
w = d.reply('import os, signal; os.kill(os.getpid(), signal.SIGKILL)', timeout=30)
with ExceptionExpected(DeadKernelError): await w
test_eq(deaths[0]['content']['execution_state'], 'dead')
test_eq((await d.model())['execution_state'], 'dead')
test_eq(await d.is_alive(), False)
await d.aclose()


`max_size` limits incoming websocket messages to 256 MiB by default. Pass `None` to disable the limit. The websocket library closes the connection when an incoming message exceeds it.


### Closing down

`shutdown` sends a `shutdown_request` on the control channel. `shutdown_kernel` uses HTTP `DELETE` instead, asking the gateway to terminate and remove the process. It also closes this client.

In [ ]:
#| export
@patch
def shutdown(self: JupyAsyncKernelClient, restart=False, timeout=None):
    return self.shutdown_request(restart=restart, channel="control", timeout=timeout)

`aclose` drains queued outbound frames before stopping the send loop. It allows up to two seconds for the drain. On a working connection, you can queue a fire-and-forget request and immediately close without discarding that request.

To delete the kernel as well as close the client, call `shutdown_kernel`.

Closing a client leaves the kernel running. Another client can attach to it and use the same namespace; messages missed during the gap are not replayed by Rustygate:

In [ ]:
a = JupyAsyncKernelClient(base_url)
await a.start_kernel()
a.start_channels()
await a.wait_for_ready(timeout=60)
await a.reply('retained = 42')
await a.aclose()


Attach the replacement client by kernel id and read the value:

In [ ]:
b = JupyAsyncKernelClient(base_url, kernel_id=a.kernel_id)
b.start_channels()
await b.wait_for_ready(timeout=60)
outs = await b.exec_outs('retained')
test_eq(outs[0]['data']['text/plain'], '42')
await b.shutdown_kernel()

### Connecting in one call

`connect` constructs a client, gets or creates a kernel, opens channels, and waits for readiness. Pass `kernel` to attach to an existing id instead. Extra keyword arguments configure kernel creation, not the client constructor.

Only a new kernel (HTTP 201) sets `owned=True`. With rustygate, `path` reuses an existing notebook kernel (HTTP 200) without claiming ownership. When used as an async context manager, the client requests deletion of an owned kernel on exit. An attached client closes its connection without deleting the kernel. Calling `aclose` directly also leaves the kernel running. Use `shutdown_kernel` when you explicitly want to delete it.

In [ ]:
#| export
@patch(cls_method=True)
async def connect(cls:JupyAsyncKernelClient, base_url, kernel=None, token=None, timeout=60, verify=True, **kw):
    "Return a ready client; only a newly created kernel is owned."
    self = cls(base_url, kernel_id=kernel, token=token, verify=verify)
    if kernel is None: await self.start_kernel(**kw)
    self.start_channels()
    await self.wait_for_ready(timeout=timeout)
    return self

@patch
async def __aenter__(self:JupyAsyncKernelClient): return self

@patch
async def __aexit__(self:JupyAsyncKernelClient, *exc):
    if self.owned:
        with suppress(Exception): await self.shutdown_kernel()
    else: await self.aclose()

In [ ]:
async with await JupyAsyncKernelClient.connect(base_url, path=Path('owned.ipynb'), cwd=Path('.')) as k2:
    kid2 = k2.kernel_id
    assert k2.owned and k2.channels_running
    async with await JupyAsyncKernelClient.connect(base_url, path='owned.ipynb') as att:
        test_eq((att.kernel_id, att.owned), (kid2, False))
    test_eq(await k2.is_alive(), True)
assert not await JupyAsyncKernelClient(base_url, kernel_id=kid2).is_alive()

With `kernel` set to an existing id, `owned` stays `False`. Leaving this block closes the attached client and leaves the kernel running:

In [ ]:
async with await JupyAsyncKernelClient.connect(base_url, kernel=kc.kernel_id) as att: assert not att.owned
test_eq(await kc.is_alive(), True)

### TLS

Rustygate's `--tls` option serves HTTPS and WSS using a self-signed certificate it generates at startup. For this disposable local test, `verify=False` disables certificate checks for both the HTTP transport and the websocket.

Disabling verification also disables server identity checks. Don't use it as a substitute for trusted certificates on an untrusted network. Token authentication still works with TLS.

In [ ]:
gs = start_gateway(tls=True)
assert gs.url.startswith('https://')
async with await JupyAsyncKernelClient.connect(gs.url, verify=False) as ks: rep = await ks.reply('6*7', timeout=30)
test_eq(rep['content']['status'], 'ok')
gs.stop()
gs

<Gateway https://127.0.0.1:61352 pid=48024 exited 0>

In [ ]:
await kc.shutdown_kernel()
test_eq(kc.channels_running, False)
test_eq(await none_kc.is_alive(), False)
await none_kc.aclose()


In [ ]:
#| hide
g.stop()


In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()